<a href="https://colab.research.google.com/github/aryan22796/AI_learning_basic/blob/main/nlp_advance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # End-to-End Restaurant Dataset Analysis and NLP Pipeline

This notebook demonstrates an end-to-end machine learning pipeline using a synthetic restaurant review dataset. It covers data generation, exploration, various NLP techniques including TF-IDF, an introduction to word embeddings and BERT, sentiment analysis, and a Logistic Regression model for classification.


 ## 1. Dataset Generation

Since no dataset was provided, we'll generate a synthetic dataset of restaurant reviews. This dataset will include unique IDs, restaurant names, review text, and a star rating (1-5).


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random

# Initialize Faker for generating realistic-looking data
faker = Faker()

# Define common positive and negative restaurant review phrases
positive_phrases = [
    "Excellent food and great service!",
    "Loved every dish, highly recommend.",
    "The atmosphere was amazing and the staff were very friendly.",
    "Best meal I've had in a long time, truly a hidden gem.",
    "Fantastic experience, will definitely be back soon.",
    "Delicious and fresh ingredients. A must-try!",
    "The ambiance is perfect for a date night.",
    "Outstanding customer service and flavorful cuisine."
]

negative_phrases = [
    "Disappointing food and slow service.",
    "The dishes were bland and overpriced.",
    "Poor experience, staff seemed uninterested.",
    "Wouldn't recommend, there are better options nearby.",
    "Waited too long for average food.",
    "The place was dirty and the food was cold.",
    "Very noisy and uncomfortable dining experience.",
    "Lack of flavor and small portions."
]

neutral_phrases = [
    "It was okay, nothing special.",
    "The food was decent, but nothing to write home about.",
    "Average experience, might come back.",
    "Standard fare, met expectations.",
    "The service was acceptable, food was fine."
]

# Generate 1000 synthetic restaurant records
def generate_restaurant_data(num_records=1000):
    data = []
    for i in range(num_records):
        restaurant_id = i + 1
        restaurant_name = faker.company() + " " + random.choice(["Cafe", "Bistro", "Grill", "Eatery", "Diner", "Kitchen"])

        # Generate ratings with a slight bias towards positive reviews
        rating = random.choices([1, 2, 3, 4, 5], weights=[0.1, 0.1, 0.2, 0.3, 0.3], k=1)[0]

        review_text = ""
        if rating == 1 or rating == 2:
            review_text = random.choice(negative_phrases) + " " + faker.paragraph(nb_sentences=1) if random.random() < 0.8 else faker.paragraph(nb_sentences=2)
        elif rating == 3:
            review_text = random.choice(neutral_phrases) + " " + faker.paragraph(nb_sentences=1) if random.random() < 0.6 else faker.paragraph(nb_sentences=2)
        else: # rating 4 or 5
            review_text = random.choice(positive_phrases) + " " + faker.paragraph(nb_sentences=1) if random.random() < 0.8 else faker.paragraph(nb_sentences=2)

        # Add some variation to review length
        if random.random() < 0.3: # make some reviews shorter
            review_text = faker.sentence(nb_words=random.randint(5, 15))
        elif random.random() > 0.7: # make some reviews longer
            review_text = faker.text(max_nb_chars=random.randint(200, 500))

        data.append({
            'restaurant_id': restaurant_id,
            'restaurant_name': restaurant_name,
            'cuisine_type': random.choice(["Italian", "Mexican", "American", "Asian", "Indian", "Vegetarian", "Seafood"]),
            'price_range': random.choice(["$", "$$", "$$$"]),
            'rating': rating,
            'review_text': review_text
        })
    return pd.DataFrame(data)

df = generate_restaurant_data(1000)

print(f"Generated dataset with {len(df)} records.")
display(df.head())


 ## 2. Data Exploration and Preprocessing

We'll perform basic exploratory data analysis (EDA) to understand the dataset's structure, identify distributions, and check for any immediate data quality issues. We'll also preprocess the text data for NLP tasks.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Display basic information about the dataset
print("Dataset Info:")
display(df.info())

print("\nDescriptive Statistics for Ratings:")
display(df['rating'].describe())

# Visualize the distribution of ratings
plt.figure(figsize=(8, 5))
sns.countplot(x='rating', data=df, palette='viridis')
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Star Rating')
plt.ylabel('Number of Reviews')
plt.show()

# Check for missing values
print("\nMissing Values:")
display(df.isnull().sum())

# Text Preprocessing Function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()  # Lowercasing
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabetic characters
    tokens = text.split()  # Tokenization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Remove stop words and lemmatize
    return ' '.join(tokens)

df['processed_review_text'] = df['review_text'].apply(preprocess_text)

display(df[['review_text', 'processed_review_text']].head())


 ## 3. Feature Engineering: NLP Techniques

This section explores various NLP techniques to convert raw text into numerical features suitable for machine learning models.

### 3.1 TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It is often used as a weighting factor in information retrieval and text mining. We'll use `TfidfVectorizer` from `sklearn` to convert our processed review text into TF-IDF features.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features for manageability

# Fit and transform the processed review text
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_review_text'])

print("TF-IDF features shape:", X_tfidf.shape)

# Display some feature names (words) generated by TF-IDF
print("\nTop 100 TF-IDF features:")
display(tfidf_vectorizer.get_feature_names_out()[:100])


 ### 3.2 Word Embeddings (Introduction)

Word embeddings are dense vector representations of words that capture semantic meaning. Words with similar meanings are represented by similar vectors in a continuous vector space. Popular methods include Word2Vec, GloVe, and FastText.

While implementing and training custom word embeddings is resource-intensive for this notebook, it's important to understand their role. For practical applications, pre-trained word embeddings (e.g., from `gensim` library) can be loaded and used to represent words.

For example, using `gensim`:

```python
# Example of loading pre-trained Word2Vec (requires downloading model first)
# from gensim.models import KeyedVectors
# word_vectors = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)
# print(word_vectors['restaurant'].shape)
```

In a full pipeline, each word in a review would be converted to its embedding vector, and these vectors could then be aggregated (e.g., averaged) per review to create a document-level embedding.


 ### 3.3 BERT (Bidirectional Encoder Representations from Transformers) (Introduction)

BERT is a powerful transformer-based language model designed to understand the context of words in text, achieving state-of-the-art results on many NLP tasks. Unlike traditional word embeddings, BERT generates contextualized embeddings, meaning the representation of a word changes based on the other words in the sentence.

Using BERT typically involves:
1.  **Tokenization:** Converting text into tokens that BERT can understand.
2.  **Generating Embeddings:** Passing tokens through a pre-trained BERT model to get contextualized word/sentence embeddings.
3.  **Fine-tuning:** Training a small additional layer on top of BERT for a specific task (e.g., sentiment classification).

For this notebook, a full BERT fine-tuning is beyond the scope due to computational resources and complexity for a single example, but we can illustrate how one might use a pre-trained BERT model for feature extraction using the `transformers` library.


In [ ]:
 # Example of using a pre-trained BERT model for feature extraction (conceptual)
# This cell requires the transformers library. You might need to install it first:
# !pip install transformers tensorflow

# from transformers import AutoTokenizer, TFAutoModel

# try:
#     tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
#     model = TFAutoModel.from_pretrained('bert-base-uncased')

#     # Example text
#     text = "This is an amazing restaurant with delicious food."
#     inputs = tokenizer(text, return_tensors='tf', padding=True, truncation=True)
#     outputs = model(inputs)

#     # The last_hidden_state contains the contextualized embeddings for each token
#     # For sentence-level features, often the [CLS] token embedding (first token) is used or pooled output
#     print("BERT output shape for last hidden state:", outputs.last_hidden_state.shape)
#     print("BERT pooled output shape (often used for classification):")
#     # Note: Not all models have a 'pooler_output'. For sequence classification, a pooling layer is added.
#     # If a model doesn't have it, you'd typically pool 'last_hidden_state' yourself.
#     # print(outputs.pooler_output.shape)

# except ImportError:
#     print("Transformers library not installed. Skipping BERT example.")
#     print("To run, uncomment '!pip install transformers tensorflow' and the code above.")
# except Exception as e:
#     print(f"Could not load BERT model: {e}")
#     print("Ensure you have internet access and necessary libraries installed.")

print("BERT introduction complete. For full implementation, consider fine-tuning with the `transformers` library.")


 ### 3.4 Sentiment Analysis

Sentiment analysis aims to determine the emotional tone behind a piece of text. We can perform sentiment analysis on our reviews to predict whether a review is positive, negative, or neutral. We'll use a simple approach here by classifying reviews based on their star rating (e.g., 1-2 stars = negative, 3 stars = neutral, 4-5 stars = positive) and later try to predict this sentiment.

For a more advanced approach, one could use pre-trained sentiment analysis models (e.g., VADER or `pipeline` from `transformers`).


In [ ]:
 # Create a target variable for sentiment: 0=Negative, 1=Neutral, 2=Positive
df['sentiment'] = df['rating'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=df, palette='pastel')
plt.title('Distribution of Sentiment Categories')
plt.xlabel('Sentiment (0=Negative, 1=Neutral, 2=Positive)')
plt.ylabel('Number of Reviews')
plt.xticks(ticks=[0, 1, 2], labels=['Negative', 'Neutral', 'Positive'])
plt.show()

y = df['sentiment']
print("Target variable (sentiment) distribution:")
display(y.value_counts())


 ## 4. Model Building: Logistic Regression

We will use Logistic Regression, a common and effective algorithm for classification tasks, to predict the sentiment of a review based on its TF-IDF features. This model will be trained on the `X_tfidf` features and the `y` (sentiment) target variable.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

# Initialize and train the Logistic Regression model
# Using 'liblinear' solver for multiclass classification and 'ovr' strategy
log_reg_model = LogisticRegression(solver='liblinear', multi_class='ovr', random_state=42, max_iter=1000)
log_reg_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")


 ## 5. Model Evaluation

After training, we evaluate the model's performance on the test set using various metrics such as accuracy, precision, recall, F1-score, and a confusion matrix.


In [ ]:
# Make predictions on the test set
y_pred = log_reg_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, target_names=['Negative', 'Neutral', 'Positive'])

print(f"Accuracy: {accuracy:.4f}")

print("\nConfusion Matrix:")
display(pd.DataFrame(conf_matrix, index=['Actual Negative', 'Actual Neutral', 'Actual Positive'],
                     columns=['Predicted Negative', 'Predicted Neutral', 'Predicted Positive']))

print("\nClassification Report:")
print(class_report)

# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Neutral', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Neutral', 'Actual Positive'])
plt.title('Confusion Matrix for Sentiment Prediction')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


 ## Conclusion

This notebook provided an end-to-end demonstration of analyzing a restaurant review dataset. We covered:

*   **Dataset Generation:** Creating a synthetic dataset with diverse reviews and ratings.
*   **Data Exploration:** Understanding the basic structure and distributions.
*   **Text Preprocessing:** Cleaning and normalizing review text.
*   **Feature Engineering (NLP):** Extracting features using TF-IDF and introducing concepts of Word Embeddings and BERT for contextual representations.
*   **Sentiment Analysis:** Defining and preparing a target variable for sentiment.
*   **Model Building:** Training a Logistic Regression classifier on TF-IDF features.
*   **Model Evaluation:** Assessing the model's performance with standard classification metrics.

This pipeline can be extended by integrating more advanced NLP models (e.g., fine-tuning BERT for sentiment), exploring different classification algorithms, or adding more sophisticated feature engineering techniques.

In [ ]:
# Install necessary libraries
!pip install faker

# Download necessary NLTK data
import nltk
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
""